In [ ]:
from imblearn.pipeline import Pipeline  
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from sklearn.base import clone
from sklearn.metrics import (f1_score, accuracy_score, recall_score,
                              classification_report, roc_auc_score,
                              balanced_accuracy_score, confusion_matrix,
                              make_scorer)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

In [ ]:
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/wdd/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

df           = data['df']
X_selected            = data['X']
y            = data['y']
groups       = data['groups']
feature_cols = data['feature_cols']

print(df.shape)
print(y.value_counts())

In [ ]:
param_dist = {
    'model__n_estimators':     [100, 200],
    'model__max_depth':        [10, None],
    'model__min_samples_leaf': [1, 4],
    'model__max_features':     ['sqrt', 0.3],
    'model__min_samples_split':[2, 5, 10],
}

scoring = {
    'F1':       make_scorer(f1_score, average='weighted', zero_division=0),
    'Accuracy': make_scorer(accuracy_score),
    'Recall':   make_scorer(recall_score, average='weighted', zero_division=0),
}

In [ ]:
selected_features = feature_cols

In [ ]:
def evaluate_best_model(best_estimator, use_smote=False):
    logo = LeaveOneGroupOut()
    all_y_test, all_y_pred, all_y_prob = [], [], []
    f1_per_group = []

    X_arr = X_selected.values if hasattr(X_selected, 'values') else np.array(X_selected)
    y_arr      = y.values if hasattr(y, 'values') else np.array(y)
    groups_arr = np.array(groups)

    for train_idx, test_idx in logo.split(X_arr, y_arr, groups_arr):
        X_train, X_test = X_arr[train_idx], X_arr[test_idx]
        y_train, y_test = y_arr[train_idx], y_arr[test_idx]

        # pré-processamento
        scaler  = StandardScaler()
        imputer = SimpleImputer(strategy='median')
        X_train = imputer.fit_transform(scaler.fit_transform(X_train))
        X_test  = imputer.transform(scaler.transform(X_test))

        # SMOTE só no treino
        if use_smote:
            X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)

        # clona o modelo com os melhores hiperparâmetros mas sem o pipeline inteiro
        model = clone(best_estimator.named_steps['model'])
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        f1_per_group.append(f1_score(y_test, y_pred, average='weighted', zero_division=0))
        all_y_test.append(y_test)
        all_y_pred.append(y_pred)
        all_y_prob.append(y_prob)

    y_test_all = np.concatenate(all_y_test)
    y_pred_all = np.concatenate(all_y_pred)
    y_prob_all = np.concatenate(all_y_prob)

    label = 'COM SMOTE' if use_smote else 'SEM SMOTE'
    print("\n" + "="*50)
    print(f"AVALIAÇÃO FINAL — {label}")
    print("="*50)
    print(classification_report(y_test_all, y_pred_all, target_names=['Baseline','Stress']))
    print(f"ROC-AUC:               {roc_auc_score(y_test_all, y_prob_all):.3f}")
    print(f"Balanced Accuracy:     {balanced_accuracy_score(y_test_all, y_pred_all):.3f}")
    print(f"F1 std entre sujeitos: {np.std(f1_per_group):.3f}")

    cm = confusion_matrix(y_test_all, y_pred_all)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Baseline', 'Stress'],
                yticklabels=['Baseline', 'Stress'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix — Random Forest ({label})')
    plt.tight_layout()
    plt.show()

    return y_test_all, y_pred_all, y_prob_all

In [ ]:
def run_search(n_iter, use_smote=False):
    logo  = LeaveOneGroupOut()
    label = 'COM SMOTE' if use_smote else 'SEM SMOTE'

    steps = [
        ('scaler',  StandardScaler()),
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   RandomForestClassifier(random_state=42, n_jobs=1)),
    ]
    if use_smote:
        # insere o SMOTE antes do modelo
        steps.insert(2, ('smote', SMOTE(random_state=42)))

    pipeline = Pipeline(steps)  # imblearn Pipeline

    print(f"\nIniciando busca {label} | n_iter={n_iter}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=scoring,
        refit='F1',
        cv=logo,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )
    search.fit(X_selected, y, groups=groups)

    print(f"\nMelhores parâmetros: {search.best_params_}")
    print(f"Melhor F1 (CV):      {search.best_score_:.3f}")

    evaluate_best_model(search.best_estimator_, use_smote=use_smote)

    return search.best_estimator_, {
        'label':          label,
        'Melhor F1':      search.best_score_,
        'Accuracy':       search.cv_results_['mean_test_Accuracy'][search.best_index_],
        'Recall':         search.cv_results_['mean_test_Recall'][search.best_index_],
        'Params':         search.best_params_,
        'best_estimator': search.best_estimator_,
    }

In [ ]:
res_com_smote = run_search(n_iter=100, use_smote=True)

In [ ]:
best_pipeline = res_com_smote[0]

rf_model = best_pipeline.named_steps['model']

importances = rf_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(20, 19))
plt.barh(np.array(selected_features)[indices], importances[indices])
plt.title("Feature Importance — Random Forest (SMOTE)")
plt.tight_layout()
plt.show()

In [ ]:
quest_cols = ['age', 'gender', 'coffee_last_hour', 'sports', 'smoker', 'ill']

quest_cols_valid = [c for c in quest_cols if c in selected_features]
quest_indices = [list(selected_features).index(c) for c in quest_cols_valid]

quest_importances = rf_model.feature_importances_[quest_indices]
indices = np.argsort(quest_importances)

plt.figure(figsize=(8, 5))
plt.barh(np.array(quest_cols_valid)[indices], quest_importances[indices])
plt.title("Feature Importance — Questionário")
plt.tight_layout()
plt.show()

In [ ]:
y_test_rf, y_pred_rf, y_prob_rf = evaluate_best_model(res_com_smote[0], use_smote=True)

roc_data_rf = {
    'y_test':   y_test_rf,
    'y_pred':   y_pred_rf,
    'y_prob':   y_prob_rf,
    'roc_auc':  roc_auc_score(y_test_rf, y_prob_rf),
    'model':    'RF',
    'smote':    True
}
os.makedirs('/home/matildxi/Secretária/Documentos/Tese/anxolotl/results/roc_data', exist_ok=True)
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/results/roc_data/rf_smote.pkl', 'wb') as f:
    pickle.dump(roc_data_rf, f)